In [ ]:
!pip install builtwith

  Preparing metadata (setup.py) ... done
  Created wheel for builtwith: filename=builtwith-1.3.4-py3-none-any.whl size=36077 sha256=28e4ea008210431ab4b915bf3ffc0753f4694e91c7eac12df7d5dc1af7cc2d29
  Stored in directory: /root/.cache/pip/wheels/7f/2d/b2/606e3df914d4aeeab99c4a4e3e9a61673d2293c2e346db00c8
Successfully built builtwith


In [ ]:
import builtwith

# Analisis teknologi yang digunakan
res = builtwith.parse('https://www.detik.com/')
print(res)

{'databases': ['Firebase'], 'advertising-networks': ['Google AdSense'], 'tag-managers': ['Google Tag Manager'], 'javascript-frameworks': ['jQuery']}


## **Crawling Data**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

HEADERS = {"User-Agent": "Mozilla/5.0"}

def crawl_detik_harian(tanggal=None, max_page=3, save_csv=True):
    if tanggal is None:
        tanggal = datetime.now().strftime("%Y/%m/%d")  # default: hari ini

    base_url = f"https://news.detik.com/indeks?date={tanggal}"
    print(f"Ambil berita tanggal: {tanggal}")

    berita_list = []
    idx = 0

    for page in range(1, max_page + 1):
        url = f"{base_url}&page={page}"
        try:
            r = requests.get(url, headers=HEADERS)
            r.raise_for_status()
            soup = BeautifulSoup(r.text, "html.parser")

            links = soup.select("article a.media__link")
            for a in links:
                idx += 1
                title = a.get_text(strip=True)
                link = a.get("href")

                # kategori dari path URL
                kategori = "-"
                if link and link.startswith("https://"):
                    parts = link.split("/")
                    if len(parts) > 3:
                        kategori = parts[3]

                # ambil isi berita
                isi = ""
                try:
                    br = requests.get(link, headers=HEADERS, timeout=5)
                    br.raise_for_status()
                    bs = BeautifulSoup(br.text, "html.parser")
                    paragraf = bs.select("div.detail__body-text p")
                    if not paragraf:
                        paragraf = bs.select("div.detail__body-text.itp_bodycontent p")
                    isi = " ".join(p.get_text(" ", strip=True) for p in paragraf[:5])
                except Exception:
                    isi = "(gagal ambil isi berita)"

                berita = {
                    "id": idx,
                    "judul": title,
                    "link": link,
                    "kategori": kategori,
                    "isi": isi
                }
                berita_list.append(berita)

                print(f"\n[{idx}] {title}")
                print(f"Kategori : {kategori}")
                print(f"Link     : {link}")
                print(f"Isi      : {isi[:150]}...")

        except Exception as e:
            print(f"Error saat akses {url}: {e}")
            continue

    # simpan ke CSV
    if save_csv and berita_list:
        df = pd.DataFrame(berita_list)
        filename = f"berita_detik_{tanggal.replace('/', '-')}.csv"
        df.to_csv(filename, index=False, encoding="utf-8")
        print(f"\n[DONE] Data {len(berita_list)} berita tersimpan di {filename}")

    return berita_list

# Contoh pemakaian: ambil semua berita hari ini
crawl_detik_harian(max_page=5)


Ambil berita tanggal: 2025/09/10

[1] 
Kategori : berita
Link     : https://news.detik.com/berita/d-8104626/cara-ikut-lelang-kpk-17-september-ini-syarat-dan-daftar-objeknya
Isi      : Komisi Pemberantasan Korupsi ( KPK ) kembali menggelar lelang barang sitaan negara pada 17 September 2025. Kegiatan ini merupakan bagian dari upaya pe...

[2] Cara Ikut Lelang KPK 17 September, Ini Syarat dan Daftar Objeknya
Kategori : berita
Link     : https://news.detik.com/berita/d-8104626/cara-ikut-lelang-kpk-17-september-ini-syarat-dan-daftar-objeknya
Isi      : Komisi Pemberantasan Korupsi ( KPK ) kembali menggelar lelang barang sitaan negara pada 17 September 2025. Kegiatan ini merupakan bagian dari upaya pe...

[3] 
Kategori : berita
Link     : https://news.detik.com/berita/d-8104622/viral-tanggul-beton-laut-cilincing-dinas-sda-dki-nyatakan-tak-keluarkan-izin
Isi      : Video menunjukkan tanggul beton di laut Cilincing , Jakarta Utara, viral di media sosial. Dinas Sumber Daya Air (SDA) DKI Jakarta

[{'id': 1,
  'judul': '',
  'link': 'https://news.detik.com/berita/d-8104626/cara-ikut-lelang-kpk-17-september-ini-syarat-dan-daftar-objeknya',
  'kategori': 'berita',
  'isi': 'Komisi Pemberantasan Korupsi ( KPK ) kembali menggelar lelang barang sitaan negara pada 17 September 2025. Kegiatan ini merupakan bagian dari upaya pemulihan aset negara hasil tindak pidana korupsi. Total ada 40 objek yang dilelang kali ini. Bagi yang hendak mengikuti lelang KPK periode 17 September ini, perlu mengetahui daftar objek yang dilelang, syarat dan ketentuan lelang, serta tata cara mengikuti lelang. Untuk mengetahuinya, simak penjelasan berikut. Dalam lelang kali ini, KPK melelang 40 objek barang rampasan yang terdiri dari kendaraan, properti, barang elektronik, dan jenis barang lainnya. Seluruh barang tersebut berasal dari tindak pidana korupsi yang telah berkekuatan hukum tetap. SCROLL TO CONTINUE WITH CONTENT Katalog objek lelang juga tersedia secara online yang dapat dicek melalui:'},
 {'id': 2,


In [ ]:
import requests
from bs4 import BeautifulSoup

def crawl_website(url):
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad status codes

        soup = BeautifulSoup(response.content, 'html.parser')

        # Ambil semua judul h1, h2, h3
        headings = soup.find_all(['h1', 'h2', 'h3'])
        for heading in headings:
            print(f"{heading.name}: {heading.get_text()}")

        # Ambil semua link
        links = soup.find_all('a', href=True)
        for link in links:
            print(f"URL: {link['href']} | Teks: {link.get_text()}")

    except requests.exceptions.RequestException as e:
        print(f"Terjadi kesalahan saat mengakses {url}: {e}")

# Gunakan fungsi
crawl_website("https://www.detik.com/")

h1: Berita Terbaru dan Terpecaya Hari ini - Detikcom
h2: 
 Flash
 
h2: 

                    Dokter AS Berhasil Transplantasi Ginjal Babi ke Manusia, Pasien Kini Bebas Dialisis                

h3: 

                                Kuat Berdikari, Ini Tips Jaga Kesehatan ala Sandiaga Uno                            

h3: 

                                Geger Mahasiswa RI Meninggal di Austria, Kenapa Serangan Panas Bisa Fatal?                            

h2: News Feed
h2: 
				Detik Pagi			
h3: 

					Kalah 0-1 dari Korsel, Timnas Indonesia U23 Gagal Lolos ke Piala Asia                

h3: 

                    Kenalin SUV Off Road Baru BYD, Harga Mulai Rp 400 Jutaan!                

h3: 

                    IHSG Pagi Ini Kembali Menguat ke 7.684, Begini Pergerakannya                

h2: Video Terpopuler
h3: 

 Video: 1.800 Seniman Hollywood Boikot Lembaga Film Israel 

h3: 

 Video iPhone 17 Resmi Diluncurkan: Warna Baru, Chip Baru! 

h3: 

 Video: AS Ngaku Dikabari Israel Sebelu

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://www.detik.com"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def crawl_detik(max_pages=1):
    data = {"id": [], "judul": [], "link": [], "kategori": [], "isi": []}
    idx = 0

    for page in range(1, max_pages + 1):
        url = f"{BASE_URL}/terpopuler?utm_source=desktop&utm_medium=terpopuler&utm_campaign=desktop_page&page={page}"
        r = requests.get(url, headers=HEADERS)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        # ✅ ambil daftar berita populer
        berita_list = soup.select("h3.media__title a")
        for berita in berita_list:
            idx += 1
            judul = berita.get_text(strip=True)
            link = berita.get("href")
            if not link.startswith("http"):
                link = BASE_URL + link

            # ✅ kategori dari URL
            kategori = "-"
            parts = link.split("/")
            if len(parts) > 3:
                kategori = parts[3]

            # ✅ ambil isi berita (5 paragraf pertama)
            isi = ""
            try:
                res = requests.get(link, headers=HEADERS, timeout=5)
                res.raise_for_status()
                soup_detail = BeautifulSoup(res.text, "html.parser")
                paragraf = soup_detail.select("div.detail__body-text.itp_bodycontent p")
                if not paragraf:
                    paragraf = soup_detail.select("div.detail__body-text p")
                isi = " ".join(p.get_text(" ", strip=True) for p in paragraf[:5])
            except Exception:
                isi = "(gagal ambil isi berita)"

            data["id"].append(idx)
            data["judul"].append(judul)
            data["link"].append(link)
            data["kategori"].append(kategori)
            data["isi"].append(isi)

            # ✅ tampilkan di console
            print(f"\n[{idx}] {judul}")
            print(f"Kategori: {kategori}")
            print(f"Link    : {link}")
            print(f"Isi     : {isi[:200]}...")

    # ✅ simpan ke CSV
    df = pd.DataFrame(data)
    df.to_csv("berita_detik.csv", index=False, encoding="utf-8")
    print(f"\n[DONE] {len(df)} berita tersimpan di berita_detik.csv")
    return df


# Contoh pemanggilan
crawl_detik(max_pages=1)



[1] Terungkap Niat Dansatsiber TNI soal Ferry Irwandi
Kategori: berita
Link    : https://news.detik.com/berita/d-8104426/terungkap-niat-dansatsiber-tni-soal-ferry-irwandi
Isi     : Komandan Satuan (Dansat) Siber TNI Brigjen Juinta Omboh Sembiring menyebut menemukan dugaan tindak pidana yang dilakukan konten kreator sekaligus pendiri Malaka Project, Ferry Irwandi . Niatan jendera...

[2] Harga Emas Antam Jatuh!
Kategori: berita-ekonomi-bisnis
Link    : https://finance.detik.com/berita-ekonomi-bisnis/d-8104521/harga-emas-antam-jatuh
Isi     : Harga emas Antam hari ini turun drastis setelah kemarin sempat pecah rekor lagi. Harga emas Antam 24 karat turun sampai Rp 12.000 per gram menjadi Rp 2.074.000 per gram, meninggalkan harga All Time Hi...

[3] Kualifikasi Piala Dunia 2026: Bolivia ke Playoff usai Tekuk Brasil
Kategori: sepakbola
Link    : https://sport.detik.com/sepakbola/bola-dunia/d-8104488/kualifikasi-piala-dunia-2026-bolivia-ke-playoff-usai-tekuk-brasil
Isi     : Bolivia mengama

,id,judul,link,kategori,isi
0,1,Terungkap Niat Dansatsiber TNI soal Ferry Irwandi,https://news.detik.com/berita/d-8104426/terung...,berita,Komandan Satuan (Dansat) Siber TNI Brigjen Jui...
1,2,Harga Emas Antam Jatuh!,https://finance.detik.com/berita-ekonomi-bisni...,berita-ekonomi-bisnis,Harga emas Antam hari ini turun drastis setela...
2,3,Kualifikasi Piala Dunia 2026: Bolivia ke Playo...,https://sport.detik.com/sepakbola/bola-dunia/d...,sepakbola,Bolivia mengamankan satu tiket ke playoff Pial...
3,4,"Demo Berdarah di Nepal, Massa Bakar Rumah dan ...",https://news.detik.com/internasional/d-8104525...,internasional,Demonstrasi berujung kericuhan berdarah pecah ...
4,5,Inggris dan Prancis Kecam Keras Serangan Israe...,https://news.detik.com/internasional/d-8104450...,internasional,Sejumlah pemimpin dunia ramai-ramai merespons ...
5,6,Ekuador Vs Argentina: Diwarnai Dua Kartu Merah...,https://sport.detik.com/sepakbola/bola-dunia/d...,sepakbola,Ekuador vs Argentina berjalan panas di laga te...
6,7,Mohanad Ali Tendang Kapten Thailand Bikin Irak...,https://sport.detik.com/sepakbola/liga-indones...,sepakbola,Mohanad Ali dikartu merah usai menendang kapte...
7,8,Terjawab Sudah Siapa Pengganti Sementara BG di...,https://news.detik.com/berita/d-8104395/terjaw...,berita,Presiden Prabowo Subianto telah menunjuk pengg...
8,9,Waspadai Tanda Kolesterol Tinggi di Dada dan C...,https://health.detik.com/berita-detikhealth/d-...,berita-detikhealth,Kolesterol adalah zat berlemak menyerupai lili...
9,10,Dokter AS Berhasil Transplantasi Ginjal Babi k...,https://health.detik.com/berita-detikhealth/d-...,berita-detikhealth,"Seorang pria di New Hampshire, Amerika Serikat..."


In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

# Konfigurasi
URL = "https://fentryca.github.io/PPW/"  # Website target
FOLDER_NAME = "gambar"             # Folder untuk simpan gambar
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

# Buat folder jika belum ada
if not os.path.exists(FOLDER_NAME):
    os.makedirs(FOLDER_NAME)
    print(f"Folder '{FOLDER_NAME}' berhasil dibuat.")

# Ambil halaman web
try:
    response = requests.get(URL, headers=HEADERS, timeout=10)
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f"Gagal mengakses {URL}: {e}")
    exit()

# Parse HTML dengan BeautifulSoup
soup = BeautifulSoup(response.content, 'html.parser')

# Cari semua tag <img>
images = soup.find_all('img')
print(f"Ditemukan {len(images)} gambar di halaman {URL}.")

# List untuk menyimpan URL gambar
image_urls = []

for i, img in enumerate(images):
    # Ambil atribut src
    src = img.get('src')
    if not src:
        print(f"Gambar ke-{i+1} tidak memiliki atribut 'src', dilewati.")
        continue

    # Konversi URL relatif ke absolut
    img_url = urljoin(URL, src)
    image_urls.append(img_url)

    # Ambil nama file dari URL
    parsed_url = urlparse(img_url)
    filename = os.path.basename(parsed_url.path)

    # Validasi nama file
    if not filename or '.' not in filename:
        ext = '.jpg'
        filename = f"image_{i+1}{ext}"

    # Pastikan ekstensi file valid
    valid_exts = ('.png', '.jpg', '.jpeg', '.gif', '.webp', '.svg')
    if not filename.lower().endswith(valid_exts):
        filename += '.jpg'

    # Path lokal
    file_path = os.path.join(FOLDER_NAME, filename)

    # Download gambar
    try:
        img_response = requests.get(img_url, headers=HEADERS, timeout=10)
        img_response.raise_for_status()

        with open(file_path, 'wb') as f:
            f.write(img_response.content)

        print(f"Berhasil download: {filename} -> {file_path}")

    except Exception as e:
        print(f"Gagal download {img_url}: {e}")

# Ringkasan
print("\nCrawling selesai!")
print(f"Semua gambar disimpan di: ./{FOLDER_NAME}")
print(f"Total gambar diproses: {len(image_urls)}")

Folder 'gambar' berhasil dibuat.
Ditemukan 0 gambar di halaman https://fentryca.github.io/PPW/.

Crawling selesai!
Semua gambar disimpan di: ./gambar
Total gambar diproses: 0
